In [ ]:
# ============================================================
# CELL 1 — Install required packages and mount Google Drive
# ============================================================

!pip install -q ftfy beautifulsoup4 pandas scikit-learn scipy joblib umap-learn transformers accelerate

from google.colab import drive
drive.mount('/content/drive')


# Cross-Domain AI vs Human Abstract Analysis — TF-IDF + UMAP and SciBERT + UMAP

This notebook follows the same 12-step pipeline shown in your workflow:

1. Corpus collection  
2. Preprocessing  
3. Source-domain feature learning on **Domain A / CS only**  
4. AI − Human contrast computation  
5. Contrast feature selection  
6. Domain A matrix construction  
7. **UMAP reduction before clustering** on Domain A  
8. Cross-domain transfer to **Domain B / Medical** using the same source-domain feature setup  
9. Domain B matrix construction  
10. **UMAP reduction before clustering** on Domain B  
11. Evaluation  
12. Final analysis  

Two representations are implemented:

- **TF-IDF + UMAP**, keeping the original vocabulary-based contrast method.
- **SciBERT + UMAP**, replacing TF-IDF features with SciBERT embedding dimensions. Because SciBERT does not produce interpretable TF-IDF terms, the “contrast vocabulary” step becomes a fixed set of **200 contrast embedding dimensions** selected from CS only: top 100 AI-dominant dimensions and top 100 Human-dominant dimensions.

In both branches, UMAP is fit on the CS source-domain feature matrix and then used to transform the Medical target-domain matrix. This keeps the cross-domain transfer rule strict and avoids learning the feature-reduction space from the target domain before evaluation.


In [ ]:
# ============================================================
# CELL 3 — Imports and file paths
# ============================================================

import os
import re
import html
import string
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from bs4 import BeautifulSoup
import ftfy

import nltk
nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# ------------------------------------------------------------
# CHANGE THESE TWO PATHS according to your Google Drive location
# ------------------------------------------------------------

CS_FILE = "/content/drive/MyDrive/cs_domain_main_cs_merged_abstracts.csv"
MED_FILE = "/content/drive/MyDrive/medi_domain_Main.csv"

# Output folder
OUTPUT_DIR = "/content/drive/MyDrive/step2_preprocessing_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

STEP2_LONG_OUTPUT = os.path.join(OUTPUT_DIR, "step2_long_corpus_cleaned.csv")
STEP2_AUDIT_OUTPUT = os.path.join(OUTPUT_DIR, "step2_preprocessing_audit.csv")
STEP2_SAMPLE_OUTPUT = os.path.join(OUTPUT_DIR, "step2_sample_preview.csv")

print("CS file exists:", os.path.exists(CS_FILE))
print("Medical file exists:", os.path.exists(MED_FILE))
print("Output folder:", OUTPUT_DIR)

In [ ]:
# ============================================================
# CELL 4 — Safe CSV reader
# ============================================================

def read_csv_safely(path):
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]

    last_error = None
    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"Loaded {os.path.basename(path)} with encoding: {enc}")
            return df, enc
        except Exception as e:
            last_error = e
            print(f"Failed {os.path.basename(path)} with {enc}: {e}")

    raise RuntimeError(f"Could not read file: {path}. Last error: {last_error}")

cs_df, cs_encoding = read_csv_safely(CS_FILE)
med_df, med_encoding = read_csv_safely(MED_FILE)

print("\nCS shape:", cs_df.shape)
print("CS columns:", cs_df.columns.tolist())

print("\nMedical shape:", med_df.shape)
print("Medical columns:", med_df.columns.tolist())


In [ ]:
# ============================================================
# CELL 5 — Basic structural cleaning and column validation
# ============================================================

# Remove empty/unnamed artifact columns if present
cs_df = cs_df.drop(columns=[c for c in cs_df.columns if str(c).startswith("Unnamed")], errors="ignore")
med_df = med_df.drop(columns=[c for c in med_df.columns if str(c).startswith("Unnamed")], errors="ignore")

# Required columns
CS_TITLE_COL = "title"
CS_HUMAN_COL = "original_abstract"
CS_AI_COL = "ai_generated_abstract"

MED_TITLE_COL = "title"
MED_HUMAN_COL = "abstract"
MED_AI_COL = "ai_generated_abstract"

required_cs = [CS_TITLE_COL, CS_HUMAN_COL, CS_AI_COL]
required_med = [MED_TITLE_COL, MED_HUMAN_COL, MED_AI_COL]

for col in required_cs:
    if col not in cs_df.columns:
        raise ValueError(f"CS file missing column: {col}")

for col in required_med:
    if col not in med_df.columns:
        raise ValueError(f"Medical file missing column: {col}")

# Fill missing values as blank strings
for col in required_cs:
    cs_df[col] = cs_df[col].fillna("").astype(str)

for col in required_med:
    med_df[col] = med_df[col].fillna("").astype(str)

print("After removing Unnamed columns:")
print("CS columns:", cs_df.columns.tolist())
print("Medical columns:", med_df.columns.tolist())

print("\nMissing check:")
print("CS missing title:", int((cs_df[CS_TITLE_COL].str.strip() == "").sum()))
print("CS missing human:", int((cs_df[CS_HUMAN_COL].str.strip() == "").sum()))
print("CS missing AI:", int((cs_df[CS_AI_COL].str.strip() == "").sum()))

print("Medical missing title:", int((med_df[MED_TITLE_COL].str.strip() == "").sum()))
print("Medical missing human:", int((med_df[MED_HUMAN_COL].str.strip() == "").sum()))
print("Medical missing AI:", int((med_df[MED_AI_COL].str.strip() == "").sum()))



In [ ]:
# ============================================================
# CELL 6 — Convert to standard long-format corpus
# ============================================================

cs_human = pd.DataFrame({
    "domain": "CS",
    "label": "Human",
    "title": cs_df[CS_TITLE_COL],
    "text_raw": cs_df[CS_HUMAN_COL],
    "source_column": CS_HUMAN_COL
})

cs_ai = pd.DataFrame({
    "domain": "CS",
    "label": "AI",
    "title": cs_df[CS_TITLE_COL],
    "text_raw": cs_df[CS_AI_COL],
    "source_column": CS_AI_COL
})

med_human = pd.DataFrame({
    "domain": "Medical",
    "label": "Human",
    "title": med_df[MED_TITLE_COL],
    "text_raw": med_df[MED_HUMAN_COL],
    "source_column": MED_HUMAN_COL
})

med_ai = pd.DataFrame({
    "domain": "Medical",
    "label": "AI",
    "title": med_df[MED_TITLE_COL],
    "text_raw": med_df[MED_AI_COL],
    "source_column": MED_AI_COL
})

corpus_df = pd.concat([cs_human, cs_ai, med_human, med_ai], ignore_index=True)

corpus_df["doc_id"] = [
    f"DOC_{i:05d}" for i in range(len(corpus_df))
]

# Reorder columns
corpus_df = corpus_df[["doc_id", "domain", "label", "title", "source_column", "text_raw"]]

print("Long-format corpus shape:", corpus_df.shape)
print("\nCounts by domain and label:")
print(corpus_df.groupby(["domain", "label"]).size())

corpus_df.head()

In [ ]:
# ============================================================
# CELL 7 — Text cleaning functions
# ============================================================

def safe_str(x):
    if pd.isna(x):
        return ""
    return str(x)

def normalize_whitespace(text):
    text = safe_str(text)
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def fix_encoding(text):
    text = safe_str(text)
    text = ftfy.fix_text(text)

    replacements = {
        "Ã¢ÂÂ": "'",
        "Ã¢ÂÂ": "'",
        "Ã¢ÂÂ": '"',
        "Ã¢ÂÂ": '"',
        "Ã¢ÂÂ": "-",
        "Ã¢ÂÂ": "-",
        "Ã¢ÂÂ¦": "...",
        "â€™": "'",
        "â€˜": "'",
        "â€œ": '"',
        "â€": '"',
        "â€“": "-",
        "â€”": "-",
        "â€¦": "...",
        "Â": "",
        "\x81": "",
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    return text

def remove_html(text):
    text = safe_str(text)
    text = html.unescape(text)
    text = re.sub(r"<[^>]*>", " ", text)
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    return text

def remove_prompt_echo(text):
    text = safe_str(text)
    text = normalize_whitespace(text)

    # Remove common leading assistant phrases
    leading_patterns = [
        r"^\s*here is .*?abstract[:\-]?\s*",
        r"^\s*here are .*?abstracts[:\-]?\s*",
        r"^\s*below is .*?abstract[:\-]?\s*",
        r"^\s*certainly[,.]?\s*",
        r"^\s*sure[,.]?\s*",
        r"^\s*of course[,.]?\s*",
        r"^\s*i can .*?[:\-]?\s*",
    ]

    for pat in leading_patterns:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)

    # Remove prompt labels
    label_patterns = [
        r"\bNew AI Generated Abstract\s*:\s*",
        r"\bAI Generated Abstract\s*:\s*",
        r"\bGenerated Abstract\s*:\s*",
        r"\bRewritten Abstract\s*:\s*",
        r"\bOriginal Abstract\s*:\s*",
    ]

    for pat in label_patterns:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)

    # Remove standalone Abstract: at beginning
    text = re.sub(r"^\s*Abstract\s*:\s*", "", text, flags=re.IGNORECASE)

    return normalize_whitespace(text)

def clean_text_basic(text, label):
    text = safe_str(text)
    text = fix_encoding(text)
    text = remove_html(text)

    if label == "AI":
        text = remove_prompt_echo(text)

    text = normalize_whitespace(text)
    return text

def contains_html(text):
    return bool(re.search(r"<[^>]+>", safe_str(text)))

def contains_encoding_artifact(text):
    bad_patterns = ["Ã", "Â", "â€", "Ã¢", "\x81"]
    text = safe_str(text)
    return any(p in text for p in bad_patterns)

def contains_prompt_echo(text):
    low = safe_str(text).lower()
    patterns = [
        "here is the abstract",
        "here is a rewritten",
        "rewritten abstract:",
        "new ai generated abstract:",
        "ai generated abstract:",
        "original abstract:",
        "generated abstract:",
    ]
    return any(p in low for p in patterns)

def word_count_raw(text):
    return len(re.findall(r"\b\w+\b", safe_str(text)))


In [ ]:
# ============================================================
# CELL 8 — Apply text cleaning and print important audit outputs
# ============================================================

tqdm.pandas()

corpus_df["text_clean"] = corpus_df.progress_apply(
    lambda row: clean_text_basic(row["text_raw"], row["label"]),
    axis=1
)

corpus_df["raw_word_count"] = corpus_df["text_raw"].apply(word_count_raw)
corpus_df["clean_word_count"] = corpus_df["text_clean"].apply(word_count_raw)

audit_before_after = corpus_df.groupby(["domain", "label"]).agg(
    docs=("doc_id", "count"),
    raw_mean_words=("raw_word_count", "mean"),
    clean_mean_words=("clean_word_count", "mean"),
    raw_min_words=("raw_word_count", "min"),
    clean_min_words=("clean_word_count", "min"),
    raw_max_words=("raw_word_count", "max"),
    clean_max_words=("clean_word_count", "max"),
).reset_index()

quality_counts = {
    "total_docs": len(corpus_df),
    "empty_text_clean": int((corpus_df["text_clean"].str.strip() == "").sum()),
    "html_remaining": int(corpus_df["text_clean"].apply(contains_html).sum()),
    "encoding_artifact_remaining": int(corpus_df["text_clean"].apply(contains_encoding_artifact).sum()),
    "prompt_echo_remaining_in_AI": int(
        corpus_df.loc[corpus_df["label"] == "AI", "text_clean"].apply(contains_prompt_echo).sum()
    ),
    "human_under_50_words": int(
        ((corpus_df["label"] == "Human") & (corpus_df["clean_word_count"] < 50)).sum()
    ),
    "ai_under_100_words": int(
        ((corpus_df["label"] == "AI") & (corpus_df["clean_word_count"] < 100)).sum()
    ),
}

print("========== STEP 2 CLEANING AUDIT ==========")
print("\nDocument counts:")
print(corpus_df.groupby(["domain", "label"]).size())

print("\nWord-count summary:")
display(audit_before_after)

print("\nQuality counts:")
for k, v in quality_counts.items():
    print(f"{k}: {v}")

print("\nSample cleaned records:")
display(corpus_df[["domain", "label", "title", "clean_word_count", "text_clean"]].head(8))


In [ ]:
# ============================================================
# CELL 9 — Tokenization, stopword removal, and lemmatization
# Fixed version: no NLTK punkt / punkt_tab dependency
# ============================================================

import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Make sure required NLTK resources exist
import nltk
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

stop_words = set(stopwords.words("english"))

# Keep useful contrast words that are normally stopwords
keep_words = {
    "not", "no", "nor",
    "against",
    "between",
    "under",
    "over",
    "more",
    "most"
}
stop_words = stop_words - keep_words

lemmatizer = WordNetLemmatizer()

def preprocess_for_tfidf(text):
    text = safe_str(text).lower()

    # Regex tokenizer: extracts alphabetic words only
    # This avoids NLTK punkt_tab error and is faster.
    tokens = re.findall(r"\b[a-zA-Z]+\b", text)

    processed = []

    for tok in tokens:
        tok = tok.lower().strip()

        if len(tok) < 2:
            continue

        if tok in stop_words:
            continue

        lemma = lemmatizer.lemmatize(tok)

        if len(lemma) < 2:
            continue

        processed.append(lemma)

    return processed

corpus_df["tokens"] = corpus_df["text_clean"].progress_apply(preprocess_for_tfidf)
corpus_df["preprocessed_text"] = corpus_df["tokens"].apply(lambda toks: " ".join(toks))
corpus_df["token_count"] = corpus_df["tokens"].apply(len)

token_audit = corpus_df.groupby(["domain", "label"]).agg(
    docs=("doc_id", "count"),
    mean_tokens=("token_count", "mean"),
    min_tokens=("token_count", "min"),
    max_tokens=("token_count", "max"),
).reset_index()

print("========== STEP 2 TOKENIZATION + LEMMATIZATION AUDIT ==========")
display(token_audit)

print("\nEmpty preprocessed documents:", int((corpus_df["preprocessed_text"].str.strip() == "").sum()))

print("\nPreview after preprocessing:")
display(corpus_df[[
    "doc_id", "domain", "label", "clean_word_count", "token_count", "preprocessed_text"
]].head(10))

In [ ]:
# ============================================================
# CELL 10 — Save Step 2 outputs
# ============================================================

audit_rows = []

for (domain, label), group in corpus_df.groupby(["domain", "label"]):
    audit_rows.append({
        "domain": domain,
        "label": label,
        "docs": len(group),
        "clean_word_mean": group["clean_word_count"].mean(),
        "clean_word_min": group["clean_word_count"].min(),
        "clean_word_max": group["clean_word_count"].max(),
        "token_mean": group["token_count"].mean(),
        "token_min": group["token_count"].min(),
        "token_max": group["token_count"].max(),
        "empty_clean_text": int((group["text_clean"].str.strip() == "").sum()),
        "empty_preprocessed_text": int((group["preprocessed_text"].str.strip() == "").sum()),
    })

step2_audit_df = pd.DataFrame(audit_rows)

sample_df = corpus_df.groupby(["domain", "label"], group_keys=False).head(3)

corpus_df.to_csv(STEP2_LONG_OUTPUT, index=False, encoding="utf-8-sig")
step2_audit_df.to_csv(STEP2_AUDIT_OUTPUT, index=False, encoding="utf-8-sig")
sample_df.to_csv(STEP2_SAMPLE_OUTPUT, index=False, encoding="utf-8-sig")

print("Saved Step 2 long corpus:", STEP2_LONG_OUTPUT)
print("Saved Step 2 audit:", STEP2_AUDIT_OUTPUT)
print("Saved Step 2 sample preview:", STEP2_SAMPLE_OUTPUT)

print("\nFinal Step 2 corpus shape:", corpus_df.shape)
print("\nFinal counts:")
print(corpus_df.groupby(["domain", "label"]).size())

In [ ]:
# ============================================================
# CELL 11 — Find remaining AI prompt-echo rows
# ============================================================

remaining_prompt_echo = corpus_df[
    (corpus_df["label"] == "AI") &
    (corpus_df["text_clean"].apply(contains_prompt_echo))
].copy()

print("Remaining AI prompt-echo rows:", len(remaining_prompt_echo))

display(
    remaining_prompt_echo[
        ["doc_id", "domain", "label", "title", "clean_word_count", "text_clean"]
    ].head(20)
)

In [ ]:
# ============================================================
# CELL 12 — Stronger final prompt-echo cleaning
# ============================================================

def remove_prompt_echo_stronger(text):
    text = safe_str(text)
    text = normalize_whitespace(text)

    # Remove very common assistant/prompt phrases at the beginning
    leading_patterns = [
        r"^\s*here is .*?abstract[:\-]?\s*",
        r"^\s*here are .*?abstracts[:\-]?\s*",
        r"^\s*below is .*?abstract[:\-]?\s*",
        r"^\s*the following is .*?abstract[:\-]?\s*",
        r"^\s*this is .*?abstract[:\-]?\s*",
        r"^\s*rewritten version[:\-]?\s*",
        r"^\s*rewritten abstract[:\-]?\s*",
        r"^\s*new ai generated abstract[:\-]?\s*",
        r"^\s*ai generated abstract[:\-]?\s*",
        r"^\s*generated abstract[:\-]?\s*",
        r"^\s*certainly[,.]?\s*",
        r"^\s*sure[,.]?\s*",
        r"^\s*of course[,.]?\s*",
    ]

    for pat in leading_patterns:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)

    # Remove prompt labels anywhere in the text
    label_patterns = [
        r"\bNew AI Generated Abstract\s*:\s*",
        r"\bAI Generated Abstract\s*:\s*",
        r"\bGenerated Abstract\s*:\s*",
        r"\bRewritten Abstract\s*:\s*",
        r"\bOriginal Abstract\s*:\s*",
        r"\bHuman Abstract\s*:\s*",
        r"\bTitle\s*:\s*",
        r"\bAbstract\s*:\s*",
    ]

    for pat in label_patterns:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)

    # Remove phrases that commonly appear in ChatGPT-style output
    phrase_patterns = [
        r"\bhere is a rewritten version\b",
        r"\bhere is the rewritten version\b",
        r"\bhere is the abstract\b",
        r"\bhere is an abstract\b",
        r"\bthe rewritten abstract is\b",
        r"\bthis rewritten abstract\b",
    ]

    for pat in phrase_patterns:
        text = re.sub(pat, "", text, flags=re.IGNORECASE)

    return normalize_whitespace(text)

# Apply only to AI rows
ai_mask = corpus_df["label"] == "AI"

before_prompt_echo = int(
    corpus_df.loc[ai_mask, "text_clean"].apply(contains_prompt_echo).sum()
)

corpus_df.loc[ai_mask, "text_clean"] = corpus_df.loc[ai_mask, "text_clean"].apply(
    remove_prompt_echo_stronger
)

corpus_df["clean_word_count"] = corpus_df["text_clean"].apply(word_count_raw)

after_prompt_echo = int(
    corpus_df.loc[ai_mask, "text_clean"].apply(contains_prompt_echo).sum()
)

print("Prompt echo before:", before_prompt_echo)
print("Prompt echo after:", after_prompt_echo)

In [ ]:
# ============================================================
# CELL 13 — Re-run tokenization after final correction
# ============================================================

corpus_df["tokens"] = corpus_df["text_clean"].progress_apply(preprocess_for_tfidf)
corpus_df["preprocessed_text"] = corpus_df["tokens"].apply(lambda toks: " ".join(toks))
corpus_df["token_count"] = corpus_df["tokens"].apply(len)

token_audit = corpus_df.groupby(["domain", "label"]).agg(
    docs=("doc_id", "count"),
    mean_tokens=("token_count", "mean"),
    min_tokens=("token_count", "min"),
    max_tokens=("token_count", "max"),
).reset_index()

print("========== FINAL STEP 2 TOKENIZATION AUDIT ==========")
display(token_audit)

print("\nEmpty preprocessed documents:", int((corpus_df["preprocessed_text"].str.strip() == "").sum()))

print("\nPrompt echo remaining in AI:", int(
    corpus_df.loc[corpus_df["label"] == "AI", "text_clean"].apply(contains_prompt_echo).sum()
))

In [ ]:
# ============================================================
# CELL 14 — Final Step 2 quality recheck
# ============================================================

final_quality_counts = {
    "total_docs": len(corpus_df),
    "empty_text_clean": int((corpus_df["text_clean"].str.strip() == "").sum()),
    "html_remaining": int(corpus_df["text_clean"].apply(contains_html).sum()),
    "encoding_artifact_remaining": int(corpus_df["text_clean"].apply(contains_encoding_artifact).sum()),
    "prompt_echo_remaining_in_AI": int(
        corpus_df.loc[corpus_df["label"] == "AI", "text_clean"].apply(contains_prompt_echo).sum()
    ),
    "human_under_50_words": int(
        ((corpus_df["label"] == "Human") & (corpus_df["clean_word_count"] < 50)).sum()
    ),
    "ai_under_100_words": int(
        ((corpus_df["label"] == "AI") & (corpus_df["clean_word_count"] < 100)).sum()
    ),
    "empty_preprocessed_documents": int((corpus_df["preprocessed_text"].str.strip() == "").sum()),
}

print("========== FINAL STEP 2 QUALITY COUNTS ==========")
for k, v in final_quality_counts.items():
    print(f"{k}: {v}")

print("\nFinal counts:")
print(corpus_df.groupby(["domain", "label"]).size())

In [ ]:
# ============================================================
# CELL 15 — Save corrected Step 2 files
# ============================================================

step2_audit_rows = []

for (domain, label), group in corpus_df.groupby(["domain", "label"]):
    step2_audit_rows.append({
        "domain": domain,
        "label": label,
        "docs": len(group),
        "clean_word_mean": group["clean_word_count"].mean(),
        "clean_word_min": group["clean_word_count"].min(),
        "clean_word_max": group["clean_word_count"].max(),
        "token_mean": group["token_count"].mean(),
        "token_min": group["token_count"].min(),
        "token_max": group["token_count"].max(),
        "empty_clean_text": int((group["text_clean"].str.strip() == "").sum()),
        "empty_preprocessed_text": int((group["preprocessed_text"].str.strip() == "").sum()),
    })

step2_audit_df = pd.DataFrame(step2_audit_rows)

sample_df = corpus_df.groupby(["domain", "label"], group_keys=False).head(3)

corpus_df.to_csv(STEP2_LONG_OUTPUT, index=False, encoding="utf-8-sig")
step2_audit_df.to_csv(STEP2_AUDIT_OUTPUT, index=False, encoding="utf-8-sig")
sample_df.to_csv(STEP2_SAMPLE_OUTPUT, index=False, encoding="utf-8-sig")

print("Saved corrected Step 2 long corpus:", STEP2_LONG_OUTPUT)
print("Saved corrected Step 2 audit:", STEP2_AUDIT_OUTPUT)
print("Saved corrected Step 2 sample preview:", STEP2_SAMPLE_OUTPUT)

print("\nFinal Step 2 corpus shape:", corpus_df.shape)
print("\nFinal counts:")
print(corpus_df.groupby(["domain", "label"]).size())

In [ ]:
# ============================================================
# CELL 16 — Inspect remaining 2 prompt-echo rows
# ============================================================

remaining_prompt_echo = corpus_df[
    (corpus_df["label"] == "AI") &
    (corpus_df["text_clean"].apply(contains_prompt_echo))
].copy()

print("Remaining AI prompt-echo rows:", len(remaining_prompt_echo))

pd.set_option("display.max_colwidth", 2000)

display(
    remaining_prompt_echo[
        ["doc_id", "domain", "label", "title", "clean_word_count", "text_clean"]
    ]
)

In [ ]:
# ============================================================
# CELL 17 — Final aggressive prompt-echo cleaning for remaining AI rows
# ============================================================

def remove_prompt_echo_final(text):
    text = safe_str(text)
    text = normalize_whitespace(text)

    # Remove assistant-style phrases anywhere, not only at the beginning
    anywhere_patterns = [
        r"\bhere is (a|the|an)?\s*.*?abstract[:\-]?\s*",
        r"\bhere are (the|some)?\s*.*?abstracts[:\-]?\s*",
        r"\bbelow is (a|the|an)?\s*.*?abstract[:\-]?\s*",
        r"\bthe following is (a|the|an)?\s*.*?abstract[:\-]?\s*",
        r"\bthis is (a|the|an)?\s*.*?abstract[:\-]?\s*",
        r"\bthe rewritten abstract is[:\-]?\s*",
        r"\bthis rewritten abstract[:\-]?\s*",
        r"\brewritten version of the abstract[:\-]?\s*",
        r"\bnew ai generated abstract[:\-]?\s*",
        r"\bai generated abstract[:\-]?\s*",
        r"\bgenerated abstract[:\-]?\s*",
        r"\brewritten abstract[:\-]?\s*",
        r"\boriginal abstract[:\-]?\s*",
        r"\bhuman abstract[:\-]?\s*",
        r"\babstract[:\-]\s*",
    ]

    for pat in anywhere_patterns:
        text = re.sub(pat, " ", text, flags=re.IGNORECASE)

    # Remove simple assistant confirmations
    confirmation_patterns = [
        r"\bcertainly[,.]?\s*",
        r"\bsure[,.]?\s*",
        r"\bof course[,.]?\s*",
    ]

    for pat in confirmation_patterns:
        text = re.sub(pat, " ", text, flags=re.IGNORECASE)

    text = normalize_whitespace(text)
    return text

# Apply only to the remaining bad AI rows
remaining_idx = corpus_df[
    (corpus_df["label"] == "AI") &
    (corpus_df["text_clean"].apply(contains_prompt_echo))
].index

print("Rows to clean:", len(remaining_idx))

corpus_df.loc[remaining_idx, "text_clean"] = corpus_df.loc[remaining_idx, "text_clean"].apply(
    remove_prompt_echo_final
)

corpus_df["clean_word_count"] = corpus_df["text_clean"].apply(word_count_raw)

print("Final aggressive cleaning applied.")

In [ ]:
# ============================================================
# CELL 18 — Re-tokenize after final prompt-echo cleaning
# ============================================================

corpus_df["tokens"] = corpus_df["text_clean"].progress_apply(preprocess_for_tfidf)
corpus_df["preprocessed_text"] = corpus_df["tokens"].apply(lambda toks: " ".join(toks))
corpus_df["token_count"] = corpus_df["tokens"].apply(len)

print("Retokenization completed.")

print("\nPrompt echo remaining in AI:", int(
    corpus_df.loc[corpus_df["label"] == "AI", "text_clean"].apply(contains_prompt_echo).sum()
))

print("Empty preprocessed documents:", int(
    (corpus_df["preprocessed_text"].str.strip() == "").sum()
))

In [ ]:
# ============================================================
# CELL 19 — Final Step 2 recheck
# ============================================================

final_quality_counts = {
    "total_docs": len(corpus_df),
    "empty_text_clean": int((corpus_df["text_clean"].str.strip() == "").sum()),
    "html_remaining": int(corpus_df["text_clean"].apply(contains_html).sum()),
    "encoding_artifact_remaining": int(corpus_df["text_clean"].apply(contains_encoding_artifact).sum()),
    "prompt_echo_remaining_in_AI": int(
        corpus_df.loc[corpus_df["label"] == "AI", "text_clean"].apply(contains_prompt_echo).sum()
    ),
    "human_under_50_words": int(
        ((corpus_df["label"] == "Human") & (corpus_df["clean_word_count"] < 50)).sum()
    ),
    "ai_under_100_words": int(
        ((corpus_df["label"] == "AI") & (corpus_df["clean_word_count"] < 100)).sum()
    ),
    "empty_preprocessed_documents": int((corpus_df["preprocessed_text"].str.strip() == "").sum()),
}

print("========== FINAL STEP 2 QUALITY COUNTS ==========")
for k, v in final_quality_counts.items():
    print(f"{k}: {v}")

print("\nFinal counts:")
print(corpus_df.groupby(["domain", "label"]).size())

In [ ]:
# ============================================================
# CELL 20 — Save final corrected Step 2 output
# ============================================================

step2_audit_rows = []

for (domain, label), group in corpus_df.groupby(["domain", "label"]):
    step2_audit_rows.append({
        "domain": domain,
        "label": label,
        "docs": len(group),
        "clean_word_mean": group["clean_word_count"].mean(),
        "clean_word_min": group["clean_word_count"].min(),
        "clean_word_max": group["clean_word_count"].max(),
        "token_mean": group["token_count"].mean(),
        "token_min": group["token_count"].min(),
        "token_max": group["token_count"].max(),
        "empty_clean_text": int((group["text_clean"].str.strip() == "").sum()),
        "empty_preprocessed_text": int((group["preprocessed_text"].str.strip() == "").sum()),
    })

step2_audit_df = pd.DataFrame(step2_audit_rows)

corpus_df.to_csv(STEP2_LONG_OUTPUT, index=False, encoding="utf-8-sig")
step2_audit_df.to_csv(STEP2_AUDIT_OUTPUT, index=False, encoding="utf-8-sig")

print("Saved final Step 2 long corpus:", STEP2_LONG_OUTPUT)
print("Saved final Step 2 audit:", STEP2_AUDIT_OUTPUT)

print("\nFinal Step 2 corpus shape:", corpus_df.shape)
print("\nFinal counts:")
print(corpus_df.groupby(["domain", "label"]).size())

In [ ]:
# ============================================================
# NEW CELL A0 — Shared setup for TF-IDF+UMAP and SciBERT+UMAP
# ============================================================

import os
import re
import json
import joblib
import math
import warnings
from itertools import combinations

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.spatial.distance import cosine

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)
from sklearn.decomposition import PCA

from umap import UMAP
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

# Keep the same contrast size used in the original TF-IDF pipeline.
TOP_AI_FEATURES = 100
TOP_HUMAN_FEATURES = 100

# UMAP setting used before clustering.
# n_components=10 is used for clustering to keep more structure than a 2D visualization.
UMAP_N_COMPONENTS_CLUSTER = 10
UMAP_N_COMPONENTS_PLOT = 2
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.0
UMAP_METRIC = "cosine"

ROOT_OUTPUT_DIR = "/content/drive/MyDrive/crossdomain_tfidf_scibert_umap_outputs"
TFIDF_UMAP_DIR = os.path.join(ROOT_OUTPUT_DIR, "tfidf_umap_pipeline")
SCIBERT_UMAP_DIR = os.path.join(ROOT_OUTPUT_DIR, "scibert_umap_pipeline")
COMPARISON_DIR = os.path.join(ROOT_OUTPUT_DIR, "final_comparison")

for d in [ROOT_OUTPUT_DIR, TFIDF_UMAP_DIR, SCIBERT_UMAP_DIR, COMPARISON_DIR]:
    os.makedirs(d, exist_ok=True)

# Load the final Step 2 file if the notebook is resumed from this point.
if "corpus_df" not in globals():
    corpus_df = pd.read_csv(STEP2_LONG_OUTPUT)

# Standardize expected columns.
required_cols = ["domain", "label", "text_clean", "preprocessed_text", "doc_id"]
missing_cols = [c for c in required_cols if c not in corpus_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns from Step 2 corpus: {missing_cols}")

corpus_df["text_clean"] = corpus_df["text_clean"].fillna("").astype(str)
corpus_df["preprocessed_text"] = corpus_df["preprocessed_text"].fillna("").astype(str)

print("Shared output folder:", ROOT_OUTPUT_DIR)
print("Corpus shape:", corpus_df.shape)
display(corpus_df.groupby(["domain", "label"]).size().reset_index(name="documents"))


In [ ]:
# ============================================================
# NEW CELL A1 — Shared utility functions
# ============================================================

def to_binary_labels(labels):
    # AI = 1, Human = 0.
    return np.array([1 if str(x).lower() == "ai" else 0 for x in labels], dtype=int)

def purity_score(y_true, y_pred):
    # Cluster purity for unsupervised evaluation.
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    total = len(y_true)
    score = 0
    for cluster_id in np.unique(y_pred):
        idx = y_pred == cluster_id
        if idx.sum() == 0:
            continue
        labels, counts = np.unique(y_true[idx], return_counts=True)
        score += counts.max()
    return score / total

def cluster_summary_df(y_true_labels, cluster_labels, domain_name, representation, algorithm):
    rows = []
    y_true_labels = np.asarray(y_true_labels)
    cluster_labels = np.asarray(cluster_labels)

    for cluster_id in sorted(np.unique(cluster_labels)):
        idx = cluster_labels == cluster_id
        group_labels = y_true_labels[idx]
        counts = pd.Series(group_labels).value_counts().to_dict()
        ai_count = int(counts.get("AI", 0))
        human_count = int(counts.get("Human", 0))
        majority = "AI" if ai_count >= human_count else "Human"
        rows.append({
            "domain": domain_name,
            "representation": representation,
            "algorithm": algorithm,
            "cluster": int(cluster_id),
            "documents": int(idx.sum()),
            "AI": ai_count,
            "Human": human_count,
            "majority_label": majority,
            "cluster_purity": max(ai_count, human_count) / max(int(idx.sum()), 1)
        })
    return pd.DataFrame(rows)

def evaluate_clustering(y_true_labels, cluster_labels, X_features, domain_name, representation, algorithm, reduction):
    y_true_bin = to_binary_labels(y_true_labels)
    cluster_labels = np.asarray(cluster_labels)

    row = {
        "domain": domain_name,
        "representation": representation,
        "reduction": reduction,
        "algorithm": algorithm,
        "documents": len(y_true_labels),
        "purity": purity_score(y_true_bin, cluster_labels),
        "ARI": adjusted_rand_score(y_true_bin, cluster_labels),
        "NMI": normalized_mutual_info_score(y_true_bin, cluster_labels),
    }

    # Internal metrics are computed on the reduced feature matrix used by clustering.
    if len(np.unique(cluster_labels)) > 1:
        try:
            row["silhouette"] = silhouette_score(X_features, cluster_labels, metric="euclidean")
        except Exception:
            row["silhouette"] = np.nan
        try:
            row["davies_bouldin"] = davies_bouldin_score(X_features, cluster_labels)
        except Exception:
            row["davies_bouldin"] = np.nan
        try:
            row["calinski_harabasz"] = calinski_harabasz_score(X_features, cluster_labels)
        except Exception:
            row["calinski_harabasz"] = np.nan
    else:
        row["silhouette"] = np.nan
        row["davies_bouldin"] = np.nan
        row["calinski_harabasz"] = np.nan

    return row

def run_clustering_algorithms(X_reduced, y_true_labels, domain_name, representation, reduction="UMAP"):
    # Run clustering algorithms after UMAP. KMeans is the primary algorithm matching the original pipeline.
    results = []
    summaries = []
    labels_by_algorithm = {}

    algorithms = {
        "KMeans": KMeans(n_clusters=2, n_init=50, random_state=RANDOM_STATE),
        "Agglomerative": AgglomerativeClustering(n_clusters=2),
        "GaussianMixture": GaussianMixture(n_components=2, covariance_type="full", n_init=10, random_state=RANDOM_STATE),
    }

    for name, model in algorithms.items():
        if name == "GaussianMixture":
            cluster_labels = model.fit_predict(X_reduced)
        else:
            cluster_labels = model.fit_predict(X_reduced)

        labels_by_algorithm[name] = cluster_labels
        results.append(evaluate_clustering(
            y_true_labels=y_true_labels,
            cluster_labels=cluster_labels,
            X_features=X_reduced,
            domain_name=domain_name,
            representation=representation,
            algorithm=name,
            reduction=reduction
        ))
        summaries.append(cluster_summary_df(
            y_true_labels=y_true_labels,
            cluster_labels=cluster_labels,
            domain_name=domain_name,
            representation=representation,
            algorithm=name
        ))

    return pd.DataFrame(results), pd.concat(summaries, ignore_index=True), labels_by_algorithm

def matrix_sparsity_report(X, name):
    if sparse.issparse(X):
        nonzero = int(X.nnz)
        total = X.shape[0] * X.shape[1]
        zero_rows = int((np.diff(X.indptr) == 0).sum()) if hasattr(X, "indptr") else int((np.asarray(X.sum(axis=1)).ravel() == 0).sum())
    else:
        nonzero = int(np.count_nonzero(X))
        total = X.size
        zero_rows = int((np.linalg.norm(X, axis=1) == 0).sum())
    return {
        "matrix": name,
        "rows": X.shape[0],
        "columns": X.shape[1],
        "nonzero_values": nonzero,
        "sparsity": 1 - (nonzero / total),
        "all_zero_rows": zero_rows
    }

def fit_umap_for_clustering(X_source, representation):
    reducer = UMAP(
        n_components=UMAP_N_COMPONENTS_CLUSTER,
        n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        metric=UMAP_METRIC,
        random_state=RANDOM_STATE,
        transform_seed=RANDOM_STATE,
    )
    X_umap = reducer.fit_transform(X_source)
    print(f"{representation} UMAP fit complete:", X_umap.shape)
    return reducer, X_umap

def fit_umap_for_plot(X_source):
    reducer = UMAP(
        n_components=UMAP_N_COMPONENTS_PLOT,
        n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=0.1,
        metric=UMAP_METRIC,
        random_state=RANDOM_STATE,
        transform_seed=RANDOM_STATE,
    )
    return reducer, reducer.fit_transform(X_source)

def save_scatter_plot(X_2d, labels, title, out_file):
    labels = np.asarray(labels)
    plt.figure(figsize=(7, 6))
    for lab in sorted(pd.unique(labels)):
        idx = labels == lab
        plt.scatter(X_2d[idx, 0], X_2d[idx, 1], s=9, alpha=0.65, label=str(lab))
    plt.title(title)
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_file, dpi=180)
    plt.show()

def is_artifact_term(term):
    # Artifact-only filtering. This is intentionally conservative.
    term = str(term).strip().lower()
    if len(term) < 2:
        return True
    if re.search(r"(http|https|www|github|doi|arxiv|html|href|nbsp|latex|tex|pdf|xml|cid|xref)", term):
        return True
    if re.search(r"[\\/<>{}\[\]\|_=+#@~^]", term):
        return True
    if re.search(r"\b[a-z]*\d+[a-z]*\b", term):
        return True
    return False

def directional_jaccard(set_a, set_b):
    set_a, set_b = set(set_a), set(set_b)
    union = set_a | set_b
    if not union:
        return np.nan
    return len(set_a & set_b) / len(union)

def npmi_coherence_from_dtm(X_dtm, term_indices):
    # Compute mean/median NPMI for selected term columns.
    if len(term_indices) < 2:
        return {"mean_npmi": np.nan, "median_npmi": np.nan, "zero_cooccurrence_pairs": np.nan}

    X_sel = X_dtm[:, term_indices]
    if sparse.issparse(X_sel):
        X_bin = (X_sel > 0).astype(np.int8).tocsr()
        doc_freq = np.asarray(X_bin.sum(axis=0)).ravel()
        cooc = (X_bin.T @ X_bin).toarray()
    else:
        X_bin = (X_sel > 0).astype(np.int8)
        doc_freq = X_bin.sum(axis=0)
        cooc = X_bin.T @ X_bin

    n_docs = X_sel.shape[0]
    values = []
    zero_pairs = 0

    for i, j in combinations(range(len(term_indices)), 2):
        p_i = doc_freq[i] / n_docs
        p_j = doc_freq[j] / n_docs
        p_ij = cooc[i, j] / n_docs
        if p_ij <= 0 or p_i <= 0 or p_j <= 0:
            zero_pairs += 1
            continue
        pmi = np.log(p_ij / (p_i * p_j))
        npmi = pmi / (-np.log(p_ij))
        values.append(npmi)

    if len(values) == 0:
        return {"mean_npmi": np.nan, "median_npmi": np.nan, "zero_cooccurrence_pairs": zero_pairs}

    return {
        "mean_npmi": float(np.mean(values)),
        "median_npmi": float(np.median(values)),
        "zero_cooccurrence_pairs": int(zero_pairs)
    }

print("Utility functions loaded.")


In [ ]:
# ============================================================
# TF-IDF BRANCH — STEP 3: TF-IDF on Domain A / CS only
# ============================================================

tfidf_cs_docs = corpus_df[corpus_df["domain"] == "CS"].copy().reset_index(drop=True)
tfidf_med_docs = corpus_df[corpus_df["domain"] == "Medical"].copy().reset_index(drop=True)

print("Domain A / CS documents:", tfidf_cs_docs.shape)
print("Domain B / Medical documents:", tfidf_med_docs.shape)
display(tfidf_cs_docs.groupby("label").size().reset_index(name="documents"))
display(tfidf_med_docs.groupby("label").size().reset_index(name="documents"))

TFIDF_CONFIG = {
    "ngram_range": (1, 2),
    "min_df": 5,
    "max_df": 0.85,
    "max_features": 30000,
    "sublinear_tf": True,
    "norm": "l2",
    "lowercase": False,
    "dtype": np.float32,
}

tfidf_vectorizer_cs = TfidfVectorizer(**TFIDF_CONFIG)
X_tfidf_cs_full = tfidf_vectorizer_cs.fit_transform(tfidf_cs_docs["preprocessed_text"].fillna(""))
tfidf_feature_names = np.array(tfidf_vectorizer_cs.get_feature_names_out())

tfidf_step3_report = pd.DataFrame([matrix_sparsity_report(X_tfidf_cs_full, "CS full TF-IDF")])
display(tfidf_step3_report)

joblib.dump(tfidf_vectorizer_cs, os.path.join(TFIDF_UMAP_DIR, "step3_tfidf_vectorizer_cs_only.joblib"))
sparse.save_npz(os.path.join(TFIDF_UMAP_DIR, "step3_cs_full_tfidf_matrix.npz"), X_tfidf_cs_full)
tfidf_cs_docs.to_csv(os.path.join(TFIDF_UMAP_DIR, "step3_cs_documents.csv"), index=False)
pd.DataFrame({"feature_id": np.arange(len(tfidf_feature_names)), "term": tfidf_feature_names}).to_csv(
    os.path.join(TFIDF_UMAP_DIR, "step3_tfidf_features.csv"), index=False
)
tfidf_step3_report.to_csv(os.path.join(TFIDF_UMAP_DIR, "step3_tfidf_report.csv"), index=False)


In [ ]:
# ============================================================
# TF-IDF BRANCH — STEP 4: Contrast computation TFIDF_AI - TFIDF_Human
# TF-IDF BRANCH — STEP 5: Contrast vocabulary selection
# ============================================================

cs_ai_mask = tfidf_cs_docs["label"].eq("AI").to_numpy()
cs_human_mask = tfidf_cs_docs["label"].eq("Human").to_numpy()

tfidf_ai_mean = np.asarray(X_tfidf_cs_full[cs_ai_mask].mean(axis=0)).ravel()
tfidf_human_mean = np.asarray(X_tfidf_cs_full[cs_human_mask].mean(axis=0)).ravel()
tfidf_contrast = tfidf_ai_mean - tfidf_human_mean

tfidf_contrast_df = pd.DataFrame({
    "feature_id": np.arange(len(tfidf_feature_names)),
    "term": tfidf_feature_names,
    "ai_mean_tfidf": tfidf_ai_mean,
    "human_mean_tfidf": tfidf_human_mean,
    "contrast_ai_minus_human": tfidf_contrast,
})
tfidf_contrast_df["direction"] = np.where(
    tfidf_contrast_df["contrast_ai_minus_human"] > 0, "AI",
    np.where(tfidf_contrast_df["contrast_ai_minus_human"] < 0, "Human", "Tie")
)

tfidf_contrast_report = pd.DataFrame([{
    "total_features": len(tfidf_contrast_df),
    "AI_dominant_features": int((tfidf_contrast_df["contrast_ai_minus_human"] > 0).sum()),
    "Human_dominant_features": int((tfidf_contrast_df["contrast_ai_minus_human"] < 0).sum()),
    "tie_features": int((tfidf_contrast_df["contrast_ai_minus_human"] == 0).sum()),
    "min_contrast": float(tfidf_contrast_df["contrast_ai_minus_human"].min()),
    "max_contrast": float(tfidf_contrast_df["contrast_ai_minus_human"].max()),
    "std_contrast": float(tfidf_contrast_df["contrast_ai_minus_human"].std()),
}])
display(tfidf_contrast_report)

# Artifact-only filtering before vocabulary selection.
tfidf_contrast_filtered = tfidf_contrast_df[~tfidf_contrast_df["term"].apply(is_artifact_term)].copy()

top_ai_tfidf = (
    tfidf_contrast_filtered[tfidf_contrast_filtered["contrast_ai_minus_human"] > 0]
    .sort_values("contrast_ai_minus_human", ascending=False)
    .head(TOP_AI_FEATURES)
    .copy()
)
top_human_tfidf = (
    tfidf_contrast_filtered[tfidf_contrast_filtered["contrast_ai_minus_human"] < 0]
    .sort_values("contrast_ai_minus_human", ascending=True)
    .head(TOP_HUMAN_FEATURES)
    .copy()
)

top_ai_tfidf["contrast_group"] = "AI"
top_human_tfidf["contrast_group"] = "Human"

tfidf_contrast_vocab_df = pd.concat([top_ai_tfidf, top_human_tfidf], ignore_index=True)
tfidf_selected_feature_ids = tfidf_contrast_vocab_df["feature_id"].astype(int).to_numpy()
tfidf_selected_terms = tfidf_contrast_vocab_df["term"].tolist()

tfidf_vocab_check = pd.DataFrame([{
    "selected_AI_terms": len(top_ai_tfidf),
    "selected_Human_terms": len(top_human_tfidf),
    "total_selected_terms": len(tfidf_contrast_vocab_df),
    "duplicates": int(tfidf_contrast_vocab_df["term"].duplicated().sum()),
    "all_AI_positive": bool((top_ai_tfidf["contrast_ai_minus_human"] > 0).all()),
    "all_Human_negative": bool((top_human_tfidf["contrast_ai_minus_human"] < 0).all()),
    "artifact_filtered_features_removed": int(len(tfidf_contrast_df) - len(tfidf_contrast_filtered)),
}])
display(tfidf_vocab_check)

print("Top AI TF-IDF contrast terms")
display(top_ai_tfidf[["term", "contrast_ai_minus_human"]].head(15))

print("Top Human TF-IDF contrast terms")
display(top_human_tfidf[["term", "contrast_ai_minus_human"]].head(15))

tfidf_contrast_df.to_csv(os.path.join(TFIDF_UMAP_DIR, "step4_tfidf_contrast_all_features.csv"), index=False)
tfidf_contrast_filtered.to_csv(os.path.join(TFIDF_UMAP_DIR, "step4_tfidf_contrast_artifact_filtered.csv"), index=False)
tfidf_contrast_vocab_df.to_csv(os.path.join(TFIDF_UMAP_DIR, "step5_tfidf_contrast_vocabulary_200.csv"), index=False)
tfidf_contrast_report.to_csv(os.path.join(TFIDF_UMAP_DIR, "step4_tfidf_contrast_report.csv"), index=False)
tfidf_vocab_check.to_csv(os.path.join(TFIDF_UMAP_DIR, "step5_tfidf_vocab_check.csv"), index=False)


In [ ]:
# ============================================================
# TF-IDF BRANCH — STEP 6: DTM construction for Domain A / CS
# TF-IDF BRANCH — STEP 7: UMAP reduction before clustering on Domain A / CS
# ============================================================

X_tfidf_cs_dtm = X_tfidf_cs_full[:, tfidf_selected_feature_ids].tocsr()

tfidf_step6_report = pd.DataFrame([matrix_sparsity_report(X_tfidf_cs_dtm, "TF-IDF CS selected 200-term DTM")])
display(tfidf_step6_report)

# Row-normalization before UMAP/clustering, matching the original normalization idea.
X_tfidf_cs_dtm_norm = normalize(X_tfidf_cs_dtm, norm="l2", axis=1, copy=True)

tfidf_umap_reducer_cluster, X_tfidf_cs_umap = fit_umap_for_clustering(
    X_tfidf_cs_dtm_norm,
    representation="TF-IDF selected contrast vocabulary"
)

tfidf_cs_eval_df, tfidf_cs_cluster_summary_df, tfidf_cs_labels_by_algorithm = run_clustering_algorithms(
    X_reduced=X_tfidf_cs_umap,
    y_true_labels=tfidf_cs_docs["label"].to_numpy(),
    domain_name="CS",
    representation="TF-IDF",
    reduction="UMAP"
)

display(tfidf_cs_eval_df)
display(tfidf_cs_cluster_summary_df)

# Separate 2D UMAP plot for visualization.
tfidf_umap_reducer_plot, X_tfidf_cs_umap_2d = fit_umap_for_plot(X_tfidf_cs_dtm_norm)
save_scatter_plot(
    X_tfidf_cs_umap_2d,
    tfidf_cs_docs["label"].to_numpy(),
    "TF-IDF + UMAP — Domain A / CS by True Label",
    os.path.join(TFIDF_UMAP_DIR, "step7_tfidf_cs_umap_true_label.png")
)
save_scatter_plot(
    X_tfidf_cs_umap_2d,
    tfidf_cs_labels_by_algorithm["KMeans"],
    "TF-IDF + UMAP — Domain A / CS by KMeans Cluster",
    os.path.join(TFIDF_UMAP_DIR, "step7_tfidf_cs_umap_kmeans_cluster.png")
)

sparse.save_npz(os.path.join(TFIDF_UMAP_DIR, "step6_tfidf_cs_selected_dtm.npz"), X_tfidf_cs_dtm)
np.save(os.path.join(TFIDF_UMAP_DIR, "step7_tfidf_cs_umap_10d.npy"), X_tfidf_cs_umap)
joblib.dump(tfidf_umap_reducer_cluster, os.path.join(TFIDF_UMAP_DIR, "step7_tfidf_umap_reducer_fit_on_cs.joblib"))
tfidf_cs_eval_df.to_csv(os.path.join(TFIDF_UMAP_DIR, "step7_tfidf_cs_umap_clustering_metrics.csv"), index=False)
tfidf_cs_cluster_summary_df.to_csv(os.path.join(TFIDF_UMAP_DIR, "step7_tfidf_cs_umap_cluster_summary.csv"), index=False)


In [ ]:
# ============================================================
# TF-IDF BRANCH — STEP 8: Apply SAME CS vocabulary setup to Medical
# TF-IDF BRANCH — STEP 9: DTM construction for Domain B / Medical
# TF-IDF BRANCH — STEP 10: UMAP reduction before clustering on Domain B / Medical
# ============================================================

# Transform only. Do not fit a new TF-IDF vectorizer on Medical.
X_tfidf_med_full = tfidf_vectorizer_cs.transform(tfidf_med_docs["preprocessed_text"].fillna(""))
X_tfidf_med_dtm = X_tfidf_med_full[:, tfidf_selected_feature_ids].tocsr()

tfidf_step8_9_report = pd.DataFrame([
    matrix_sparsity_report(X_tfidf_med_full, "Medical full TF-IDF using CS vectorizer"),
    matrix_sparsity_report(X_tfidf_med_dtm, "Medical selected 200-term DTM using CS vocabulary"),
])
display(tfidf_step8_9_report)

# Normalize, then use the CS-fitted UMAP reducer to transform the Medical matrix.
# This preserves the cross-domain transfer rule and avoids fitting UMAP on the target domain.
X_tfidf_med_dtm_norm = normalize(X_tfidf_med_dtm, norm="l2", axis=1, copy=True)
X_tfidf_med_umap = tfidf_umap_reducer_cluster.transform(X_tfidf_med_dtm_norm)

tfidf_med_eval_df, tfidf_med_cluster_summary_df, tfidf_med_labels_by_algorithm = run_clustering_algorithms(
    X_reduced=X_tfidf_med_umap,
    y_true_labels=tfidf_med_docs["label"].to_numpy(),
    domain_name="Medical",
    representation="TF-IDF",
    reduction="CS-fitted UMAP"
)

display(tfidf_med_eval_df)
display(tfidf_med_cluster_summary_df)

# Medical 2D plot uses the CS-fitted 2D reducer for visualization transfer.
X_tfidf_med_umap_2d = tfidf_umap_reducer_plot.transform(X_tfidf_med_dtm_norm)
save_scatter_plot(
    X_tfidf_med_umap_2d,
    tfidf_med_docs["label"].to_numpy(),
    "TF-IDF + CS-fitted UMAP — Domain B / Medical by True Label",
    os.path.join(TFIDF_UMAP_DIR, "step10_tfidf_medical_umap_true_label.png")
)
save_scatter_plot(
    X_tfidf_med_umap_2d,
    tfidf_med_labels_by_algorithm["KMeans"],
    "TF-IDF + CS-fitted UMAP — Domain B / Medical by KMeans Cluster",
    os.path.join(TFIDF_UMAP_DIR, "step10_tfidf_medical_umap_kmeans_cluster.png")
)

tfidf_med_docs.to_csv(os.path.join(TFIDF_UMAP_DIR, "step8_medical_documents.csv"), index=False)
sparse.save_npz(os.path.join(TFIDF_UMAP_DIR, "step8_medical_full_tfidf_using_cs_vectorizer.npz"), X_tfidf_med_full)
sparse.save_npz(os.path.join(TFIDF_UMAP_DIR, "step9_tfidf_medical_selected_dtm.npz"), X_tfidf_med_dtm)
np.save(os.path.join(TFIDF_UMAP_DIR, "step10_tfidf_medical_umap_10d.npy"), X_tfidf_med_umap)
tfidf_step8_9_report.to_csv(os.path.join(TFIDF_UMAP_DIR, "step8_9_tfidf_medical_transfer_report.csv"), index=False)
tfidf_med_eval_df.to_csv(os.path.join(TFIDF_UMAP_DIR, "step10_tfidf_medical_umap_clustering_metrics.csv"), index=False)
tfidf_med_cluster_summary_df.to_csv(os.path.join(TFIDF_UMAP_DIR, "step10_tfidf_medical_umap_cluster_summary.csv"), index=False)


In [ ]:
# ============================================================
# TF-IDF BRANCH — STEP 11: Evaluation
# TF-IDF BRANCH — STEP 12: Analysis
# ============================================================

# Main clustering evaluation table.
tfidf_main_eval = pd.concat([tfidf_cs_eval_df, tfidf_med_eval_df], ignore_index=True)
display(tfidf_main_eval)

# Directional transfer: evaluate whether CS-selected terms keep the same AI/Human direction in Medical.
med_ai_mask = tfidf_med_docs["label"].eq("AI").to_numpy()
med_human_mask = tfidf_med_docs["label"].eq("Human").to_numpy()

tfidf_med_ai_mean_selected = np.asarray(X_tfidf_med_dtm[med_ai_mask].mean(axis=0)).ravel()
tfidf_med_human_mean_selected = np.asarray(X_tfidf_med_dtm[med_human_mask].mean(axis=0)).ravel()
tfidf_med_contrast_selected = tfidf_med_ai_mean_selected - tfidf_med_human_mean_selected

tfidf_cs_contrast_selected = tfidf_contrast_vocab_df["contrast_ai_minus_human"].to_numpy()
tfidf_selected_groups = tfidf_contrast_vocab_df["contrast_group"].to_numpy()

tfidf_sign_agreement = np.sign(tfidf_cs_contrast_selected) == np.sign(tfidf_med_contrast_selected)

tfidf_stability_report = pd.DataFrame([{
    "representation": "TF-IDF",
    "selected_terms": len(tfidf_selected_terms),
    "selected_terms_present_in_medical": int((np.asarray(X_tfidf_med_dtm.sum(axis=0)).ravel() > 0).sum()),
    "presence_jaccard_all_selected_terms": float((np.asarray(X_tfidf_med_dtm.sum(axis=0)).ravel() > 0).sum() / len(tfidf_selected_terms)),
    "overall_direction_sign_agreement": float(tfidf_sign_agreement.mean()),
    "AI_vocab_direction_agreement": float(tfidf_sign_agreement[tfidf_selected_groups == "AI"].mean()),
    "Human_vocab_direction_agreement": float(tfidf_sign_agreement[tfidf_selected_groups == "Human"].mean()),
}])
display(tfidf_stability_report)

# Vocabulary coherence with NPMI.
ai_term_positions = list(range(TOP_AI_FEATURES))
human_term_positions = list(range(TOP_AI_FEATURES, TOP_AI_FEATURES + TOP_HUMAN_FEATURES))
all_term_positions = list(range(TOP_AI_FEATURES + TOP_HUMAN_FEATURES))

tfidf_coherence_rows = []
for domain_name, X_dtm in [("CS", X_tfidf_cs_dtm), ("Medical", X_tfidf_med_dtm)]:
    for vocab_name, idxs in [
        ("AI contrast vocab", ai_term_positions),
        ("Human contrast vocab", human_term_positions),
        ("All contrast vocab", all_term_positions),
    ]:
        row = {"domain": domain_name, "vocabulary_group": vocab_name}
        row.update(npmi_coherence_from_dtm(X_dtm, idxs))
        tfidf_coherence_rows.append(row)

tfidf_coherence_df = pd.DataFrame(tfidf_coherence_rows)
display(tfidf_coherence_df)

# Written interpretation file.
tfidf_analysis_text = f"""
TF-IDF + UMAP cross-domain analysis

Method:
- TF-IDF was fit on Domain A / CS only.
- Contrast was computed as mean(TFIDF_AI) - mean(TFIDF_Human).
- The top {TOP_AI_FEATURES} AI-dominant and top {TOP_HUMAN_FEATURES} Human-dominant terms were selected from CS only.
- UMAP was fit on the CS selected-term DTM and then used to transform Medical.
- Clustering was performed after UMAP reduction.

Main results:
{tfidf_main_eval.to_string(index=False)}

Stability:
{tfidf_stability_report.to_string(index=False)}

Coherence:
{tfidf_coherence_df.to_string(index=False)}
""".strip()

with open(os.path.join(TFIDF_UMAP_DIR, "step12_tfidf_umap_analysis.txt"), "w", encoding="utf-8") as f:
    f.write(tfidf_analysis_text)

tfidf_main_eval.to_csv(os.path.join(TFIDF_UMAP_DIR, "step11_tfidf_umap_main_evaluation.csv"), index=False)
tfidf_stability_report.to_csv(os.path.join(TFIDF_UMAP_DIR, "step11_tfidf_vocab_stability.csv"), index=False)
tfidf_coherence_df.to_csv(os.path.join(TFIDF_UMAP_DIR, "step11_tfidf_npmi_coherence.csv"), index=False)

print(tfidf_analysis_text)


In [ ]:
# ============================================================
# SciBERT BRANCH — STEP 3 setup: Load SciBERT encoder
# ============================================================

import torch
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm

SCIBERT_MODEL_NAME = "allenai/scibert_scivocab_uncased"
SCIBERT_MAX_LENGTH = 512
SCIBERT_BATCH_SIZE = 16

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(SCIBERT_MODEL_NAME)
scibert_model = AutoModel.from_pretrained(SCIBERT_MODEL_NAME).to(device)
scibert_model.eval()

def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def encode_with_scibert(texts, output_file, batch_size=SCIBERT_BATCH_SIZE, max_length=SCIBERT_MAX_LENGTH):
    # Encode cleaned abstracts with SciBERT mean pooling. Existing saved embeddings are reused automatically.
    if os.path.exists(output_file):
        print("Loading cached SciBERT embeddings:", output_file)
        return np.load(output_file)

    embeddings = []
    texts = [str(t) if pd.notna(t) else "" for t in texts]

    for start in tqdm(range(0, len(texts), batch_size), desc=f"Encoding {os.path.basename(output_file)}"):
        batch_texts = texts[start:start + batch_size]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt"
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}

        with torch.no_grad():
            outputs = scibert_model(**encoded)
            pooled = mean_pool(outputs.last_hidden_state, encoded["attention_mask"])
            pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)

        embeddings.append(pooled.detach().cpu().numpy())

    embeddings = np.vstack(embeddings).astype(np.float32)
    np.save(output_file, embeddings)
    print("Saved embeddings:", output_file, embeddings.shape)
    return embeddings

print("SciBERT model loaded:", SCIBERT_MODEL_NAME)


In [ ]:
# ============================================================
# SciBERT BRANCH — STEP 3: SciBERT embeddings on Domain A / CS only
# SciBERT BRANCH — STEP 4: Contrast computation Embedding_AI - Embedding_Human
# SciBERT BRANCH — STEP 5: Contrast feature selection
# ============================================================

scibert_cs_docs = corpus_df[corpus_df["domain"] == "CS"].copy().reset_index(drop=True)
scibert_med_docs = corpus_df[corpus_df["domain"] == "Medical"].copy().reset_index(drop=True)

# SciBERT should use cleaned natural text, not lemmatized TF-IDF text.
X_scibert_cs_full = encode_with_scibert(
    scibert_cs_docs["text_clean"].fillna("").tolist(),
    os.path.join(SCIBERT_UMAP_DIR, "step3_scibert_cs_embeddings.npy")
)

scibert_step3_report = pd.DataFrame([{
    "matrix": "CS SciBERT embeddings",
    "rows": X_scibert_cs_full.shape[0],
    "columns": X_scibert_cs_full.shape[1],
    "embedding_model": SCIBERT_MODEL_NAME,
    "max_length": SCIBERT_MAX_LENGTH,
    "batch_size": SCIBERT_BATCH_SIZE,
    "device": device,
}])
display(scibert_step3_report)

scibert_cs_ai_mask = scibert_cs_docs["label"].eq("AI").to_numpy()
scibert_cs_human_mask = scibert_cs_docs["label"].eq("Human").to_numpy()

scibert_ai_mean = X_scibert_cs_full[scibert_cs_ai_mask].mean(axis=0)
scibert_human_mean = X_scibert_cs_full[scibert_cs_human_mask].mean(axis=0)
scibert_contrast = scibert_ai_mean - scibert_human_mean

scibert_contrast_df = pd.DataFrame({
    "feature_id": np.arange(X_scibert_cs_full.shape[1]),
    "feature_name": [f"scibert_dim_{i}" for i in range(X_scibert_cs_full.shape[1])],
    "ai_mean_embedding_value": scibert_ai_mean,
    "human_mean_embedding_value": scibert_human_mean,
    "contrast_ai_minus_human": scibert_contrast,
})
scibert_contrast_df["direction"] = np.where(
    scibert_contrast_df["contrast_ai_minus_human"] > 0, "AI",
    np.where(scibert_contrast_df["contrast_ai_minus_human"] < 0, "Human", "Tie")
)

top_ai_scibert = (
    scibert_contrast_df[scibert_contrast_df["contrast_ai_minus_human"] > 0]
    .sort_values("contrast_ai_minus_human", ascending=False)
    .head(TOP_AI_FEATURES)
    .copy()
)
top_human_scibert = (
    scibert_contrast_df[scibert_contrast_df["contrast_ai_minus_human"] < 0]
    .sort_values("contrast_ai_minus_human", ascending=True)
    .head(TOP_HUMAN_FEATURES)
    .copy()
)

top_ai_scibert["contrast_group"] = "AI"
top_human_scibert["contrast_group"] = "Human"

scibert_contrast_feature_df = pd.concat([top_ai_scibert, top_human_scibert], ignore_index=True)
scibert_selected_feature_ids = scibert_contrast_feature_df["feature_id"].astype(int).to_numpy()

scibert_contrast_report = pd.DataFrame([{
    "total_embedding_dimensions": X_scibert_cs_full.shape[1],
    "AI_dominant_dimensions": int((scibert_contrast_df["contrast_ai_minus_human"] > 0).sum()),
    "Human_dominant_dimensions": int((scibert_contrast_df["contrast_ai_minus_human"] < 0).sum()),
    "selected_AI_dimensions": len(top_ai_scibert),
    "selected_Human_dimensions": len(top_human_scibert),
    "total_selected_dimensions": len(scibert_contrast_feature_df),
    "all_AI_positive": bool((top_ai_scibert["contrast_ai_minus_human"] > 0).all()),
    "all_Human_negative": bool((top_human_scibert["contrast_ai_minus_human"] < 0).all()),
}])
display(scibert_contrast_report)

print("Top AI SciBERT contrast dimensions")
display(top_ai_scibert[["feature_name", "contrast_ai_minus_human"]].head(15))

print("Top Human SciBERT contrast dimensions")
display(top_human_scibert[["feature_name", "contrast_ai_minus_human"]].head(15))

scibert_cs_docs.to_csv(os.path.join(SCIBERT_UMAP_DIR, "step3_scibert_cs_documents.csv"), index=False)
scibert_step3_report.to_csv(os.path.join(SCIBERT_UMAP_DIR, "step3_scibert_report.csv"), index=False)
scibert_contrast_df.to_csv(os.path.join(SCIBERT_UMAP_DIR, "step4_scibert_contrast_all_dimensions.csv"), index=False)
scibert_contrast_feature_df.to_csv(os.path.join(SCIBERT_UMAP_DIR, "step5_scibert_contrast_dimensions_200.csv"), index=False)
scibert_contrast_report.to_csv(os.path.join(SCIBERT_UMAP_DIR, "step4_5_scibert_contrast_report.csv"), index=False)


In [ ]:
# ============================================================
# SciBERT BRANCH — STEP 6: Matrix construction for Domain A / CS
# SciBERT BRANCH — STEP 7: UMAP reduction before clustering on Domain A / CS
# ============================================================

X_scibert_cs_selected = X_scibert_cs_full[:, scibert_selected_feature_ids]

scibert_step6_report = pd.DataFrame([matrix_sparsity_report(X_scibert_cs_selected, "SciBERT CS selected 200-dimension matrix")])
display(scibert_step6_report)

# L2 normalization after selecting contrast dimensions.
X_scibert_cs_selected_norm = normalize(X_scibert_cs_selected, norm="l2", axis=1, copy=True)

scibert_umap_reducer_cluster, X_scibert_cs_umap = fit_umap_for_clustering(
    X_scibert_cs_selected_norm,
    representation="SciBERT selected contrast dimensions"
)

scibert_cs_eval_df, scibert_cs_cluster_summary_df, scibert_cs_labels_by_algorithm = run_clustering_algorithms(
    X_reduced=X_scibert_cs_umap,
    y_true_labels=scibert_cs_docs["label"].to_numpy(),
    domain_name="CS",
    representation="SciBERT",
    reduction="UMAP"
)

display(scibert_cs_eval_df)
display(scibert_cs_cluster_summary_df)

scibert_umap_reducer_plot, X_scibert_cs_umap_2d = fit_umap_for_plot(X_scibert_cs_selected_norm)
save_scatter_plot(
    X_scibert_cs_umap_2d,
    scibert_cs_docs["label"].to_numpy(),
    "SciBERT + UMAP — Domain A / CS by True Label",
    os.path.join(SCIBERT_UMAP_DIR, "step7_scibert_cs_umap_true_label.png")
)
save_scatter_plot(
    X_scibert_cs_umap_2d,
    scibert_cs_labels_by_algorithm["KMeans"],
    "SciBERT + UMAP — Domain A / CS by KMeans Cluster",
    os.path.join(SCIBERT_UMAP_DIR, "step7_scibert_cs_umap_kmeans_cluster.png")
)

np.save(os.path.join(SCIBERT_UMAP_DIR, "step6_scibert_cs_selected_matrix.npy"), X_scibert_cs_selected)
np.save(os.path.join(SCIBERT_UMAP_DIR, "step7_scibert_cs_umap_10d.npy"), X_scibert_cs_umap)
joblib.dump(scibert_umap_reducer_cluster, os.path.join(SCIBERT_UMAP_DIR, "step7_scibert_umap_reducer_fit_on_cs.joblib"))
scibert_cs_eval_df.to_csv(os.path.join(SCIBERT_UMAP_DIR, "step7_scibert_cs_umap_clustering_metrics.csv"), index=False)
scibert_cs_cluster_summary_df.to_csv(os.path.join(SCIBERT_UMAP_DIR, "step7_scibert_cs_umap_cluster_summary.csv"), index=False)


In [ ]:
# ============================================================
# SciBERT BRANCH — STEP 8: Apply SAME SciBERT contrast feature IDs to Medical
# SciBERT BRANCH — STEP 9: Matrix construction for Domain B / Medical
# SciBERT BRANCH — STEP 10: UMAP reduction before clustering on Domain B / Medical
# ============================================================

X_scibert_med_full = encode_with_scibert(
    scibert_med_docs["text_clean"].fillna("").tolist(),
    os.path.join(SCIBERT_UMAP_DIR, "step8_scibert_medical_embeddings.npy")
)

# Use the exact same CS-selected contrast embedding dimensions.
X_scibert_med_selected = X_scibert_med_full[:, scibert_selected_feature_ids]

scibert_step8_9_report = pd.DataFrame([
    {
        "matrix": "Medical SciBERT full embeddings",
        "rows": X_scibert_med_full.shape[0],
        "columns": X_scibert_med_full.shape[1],
        "embedding_model": SCIBERT_MODEL_NAME,
        "CS_selected_feature_mismatch": int(X_scibert_med_full.shape[1] != X_scibert_cs_full.shape[1]),
    },
    matrix_sparsity_report(X_scibert_med_selected, "Medical SciBERT selected 200-dimension matrix")
])
display(scibert_step8_9_report)

X_scibert_med_selected_norm = normalize(X_scibert_med_selected, norm="l2", axis=1, copy=True)

# Use the CS-fitted UMAP reducer for Medical transfer.
X_scibert_med_umap = scibert_umap_reducer_cluster.transform(X_scibert_med_selected_norm)

scibert_med_eval_df, scibert_med_cluster_summary_df, scibert_med_labels_by_algorithm = run_clustering_algorithms(
    X_reduced=X_scibert_med_umap,
    y_true_labels=scibert_med_docs["label"].to_numpy(),
    domain_name="Medical",
    representation="SciBERT",
    reduction="CS-fitted UMAP"
)

display(scibert_med_eval_df)
display(scibert_med_cluster_summary_df)

X_scibert_med_umap_2d = scibert_umap_reducer_plot.transform(X_scibert_med_selected_norm)
save_scatter_plot(
    X_scibert_med_umap_2d,
    scibert_med_docs["label"].to_numpy(),
    "SciBERT + CS-fitted UMAP — Domain B / Medical by True Label",
    os.path.join(SCIBERT_UMAP_DIR, "step10_scibert_medical_umap_true_label.png")
)
save_scatter_plot(
    X_scibert_med_umap_2d,
    scibert_med_labels_by_algorithm["KMeans"],
    "SciBERT + CS-fitted UMAP — Domain B / Medical by KMeans Cluster",
    os.path.join(SCIBERT_UMAP_DIR, "step10_scibert_medical_umap_kmeans_cluster.png")
)

scibert_med_docs.to_csv(os.path.join(SCIBERT_UMAP_DIR, "step8_scibert_medical_documents.csv"), index=False)
np.save(os.path.join(SCIBERT_UMAP_DIR, "step9_scibert_medical_selected_matrix.npy"), X_scibert_med_selected)
np.save(os.path.join(SCIBERT_UMAP_DIR, "step10_scibert_medical_umap_10d.npy"), X_scibert_med_umap)
scibert_step8_9_report.to_csv(os.path.join(SCIBERT_UMAP_DIR, "step8_9_scibert_medical_transfer_report.csv"), index=False)
scibert_med_eval_df.to_csv(os.path.join(SCIBERT_UMAP_DIR, "step10_scibert_medical_umap_clustering_metrics.csv"), index=False)
scibert_med_cluster_summary_df.to_csv(os.path.join(SCIBERT_UMAP_DIR, "step10_scibert_medical_umap_cluster_summary.csv"), index=False)


In [ ]:
# ============================================================
# SciBERT BRANCH — STEP 11: Evaluation
# SciBERT BRANCH — STEP 12: Analysis
# ============================================================

scibert_main_eval = pd.concat([scibert_cs_eval_df, scibert_med_eval_df], ignore_index=True)
display(scibert_main_eval)

# Directional transfer for SciBERT dimensions.
scibert_med_ai_mask = scibert_med_docs["label"].eq("AI").to_numpy()
scibert_med_human_mask = scibert_med_docs["label"].eq("Human").to_numpy()

scibert_med_ai_mean = X_scibert_med_full[scibert_med_ai_mask].mean(axis=0)
scibert_med_human_mean = X_scibert_med_full[scibert_med_human_mask].mean(axis=0)
scibert_med_contrast = scibert_med_ai_mean - scibert_med_human_mean

scibert_selected_groups = scibert_contrast_feature_df["contrast_group"].to_numpy()
scibert_cs_contrast_selected = scibert_contrast[scibert_selected_feature_ids]
scibert_med_contrast_selected = scibert_med_contrast[scibert_selected_feature_ids]
scibert_sign_agreement = np.sign(scibert_cs_contrast_selected) == np.sign(scibert_med_contrast_selected)

# Directional Jaccard compares top CS dimensions with top Medical dimensions for interpretive stability.
medical_contrast_df = pd.DataFrame({
    "feature_id": np.arange(len(scibert_med_contrast)),
    "contrast_ai_minus_human": scibert_med_contrast
})
medical_top_ai_dims = set(
    medical_contrast_df[medical_contrast_df["contrast_ai_minus_human"] > 0]
    .sort_values("contrast_ai_minus_human", ascending=False)
    .head(TOP_AI_FEATURES)["feature_id"].astype(int)
)
medical_top_human_dims = set(
    medical_contrast_df[medical_contrast_df["contrast_ai_minus_human"] < 0]
    .sort_values("contrast_ai_minus_human", ascending=True)
    .head(TOP_HUMAN_FEATURES)["feature_id"].astype(int)
)

cs_top_ai_dims = set(top_ai_scibert["feature_id"].astype(int))
cs_top_human_dims = set(top_human_scibert["feature_id"].astype(int))

scibert_stability_report = pd.DataFrame([{
    "representation": "SciBERT",
    "selected_dimensions": len(scibert_selected_feature_ids),
    "overall_direction_sign_agreement": float(scibert_sign_agreement.mean()),
    "AI_dimension_direction_agreement": float(scibert_sign_agreement[scibert_selected_groups == "AI"].mean()),
    "Human_dimension_direction_agreement": float(scibert_sign_agreement[scibert_selected_groups == "Human"].mean()),
    "directional_jaccard_CS_AI_dims_vs_Medical_AI_dims": directional_jaccard(cs_top_ai_dims, medical_top_ai_dims),
    "directional_jaccard_CS_Human_dims_vs_Medical_Human_dims": directional_jaccard(cs_top_human_dims, medical_top_human_dims),
    "cosine_similarity_full_CS_and_Medical_contrast_vectors": float(1 - cosine(scibert_contrast, scibert_med_contrast)),
}])
display(scibert_stability_report)

scibert_analysis_text = f"""
SciBERT + UMAP cross-domain analysis

Method:
- SciBERT ({SCIBERT_MODEL_NAME}) was used to encode cleaned abstracts.
- Embeddings were generated for Domain A / CS and Domain B / Medical.
- Contrast was computed on CS as mean(SciBERT_AI) - mean(SciBERT_Human) for each embedding dimension.
- The top {TOP_AI_FEATURES} AI-dominant and top {TOP_HUMAN_FEATURES} Human-dominant embedding dimensions were selected from CS only.
- The same selected SciBERT dimensions were applied to Medical.
- UMAP was fit on CS selected SciBERT dimensions and then used to transform Medical.
- Clustering was performed after UMAP reduction.

Important note:
- SciBERT does not produce an interpretable TF-IDF vocabulary. Therefore the Step 5 "contrast vocabulary" is implemented as a "contrast feature set" of embedding dimensions.
- NPMI vocabulary coherence is not applicable to SciBERT dimensions, so stability is measured using sign agreement, directional Jaccard, and contrast-vector cosine similarity.

Main results:
{scibert_main_eval.to_string(index=False)}

Stability:
{scibert_stability_report.to_string(index=False)}
""".strip()

with open(os.path.join(SCIBERT_UMAP_DIR, "step12_scibert_umap_analysis.txt"), "w", encoding="utf-8") as f:
    f.write(scibert_analysis_text)

scibert_main_eval.to_csv(os.path.join(SCIBERT_UMAP_DIR, "step11_scibert_umap_main_evaluation.csv"), index=False)
scibert_stability_report.to_csv(os.path.join(SCIBERT_UMAP_DIR, "step11_scibert_directional_stability.csv"), index=False)

print(scibert_analysis_text)


In [ ]:
# ============================================================
# FINAL COMPARISON — TF-IDF + UMAP vs SciBERT + UMAP
# ============================================================

final_eval_comparison = pd.concat([tfidf_main_eval, scibert_main_eval], ignore_index=True)
display(final_eval_comparison)

# Primary comparison: KMeans only, because KMeans is the original clustering algorithm in the earlier notebook.
primary_kmeans_comparison = final_eval_comparison[final_eval_comparison["algorithm"] == "KMeans"].copy()
display(primary_kmeans_comparison)

final_stability_comparison = pd.concat([
    tfidf_stability_report.assign(stability_type="vocabulary_terms"),
    scibert_stability_report.assign(stability_type="embedding_dimensions")
], ignore_index=True, sort=False)
display(final_stability_comparison)

final_eval_comparison.to_csv(os.path.join(COMPARISON_DIR, "final_all_clustering_metrics_tfidf_vs_scibert_umap.csv"), index=False)
primary_kmeans_comparison.to_csv(os.path.join(COMPARISON_DIR, "final_primary_kmeans_metrics_tfidf_vs_scibert_umap.csv"), index=False)
final_stability_comparison.to_csv(os.path.join(COMPARISON_DIR, "final_stability_tfidf_vs_scibert_umap.csv"), index=False)

summary_text = f"""
Final comparison: TF-IDF + UMAP vs SciBERT + UMAP

Pipeline rule followed:
- Domain A / CS is used for feature learning, contrast computation, contrast feature selection, and UMAP fitting.
- Domain B / Medical is transformed with the source-domain TF-IDF/SciBERT feature setup and the CS-fitted UMAP reducer.
- Clustering is evaluated with Purity, ARI, and NMI, with KMeans treated as the primary algorithm because it matches the original notebook.

Primary KMeans comparison:
{primary_kmeans_comparison.to_string(index=False)}

All clustering algorithms:
{final_eval_comparison.to_string(index=False)}

Stability comparison:
{final_stability_comparison.to_string(index=False)}

Interpretation guide:
- If TF-IDF + UMAP remains strong, the original contrast vocabulary is robust even after nonlinear dimensionality reduction.
- If SciBERT + UMAP is stronger, semantic embedding features capture broader AI/Human writing signals beyond surface vocabulary.
- If TF-IDF outperforms SciBERT, explicit contrast words are more stable than dense embedding dimensions for this task.
- Compare CS and Medical scores to judge cross-domain stability.
""".strip()

with open(os.path.join(COMPARISON_DIR, "final_tfidf_vs_scibert_umap_summary.txt"), "w", encoding="utf-8") as f:
    f.write(summary_text)

print(summary_text)
print("\nAll outputs saved to:", ROOT_OUTPUT_DIR)
